# Bangla ASR test: AI4Bharat IndicConformer-600M (Bengali)

Multilingual **Conformer, hybrid CTC + RNNT**, 600M params, 22 Indian languages incl. Bengali (`bn`).
**License: MIT (commercial use OK).** Runs with **ONNX Runtime**, no NeMo needed.

Two downloads of the same model:
| `VARIANT` | Repo | Size | Notes |
|---|---|---|---|
| `int8` | `yashwork-byte/indic-conformer-600m-int8-onnx` | ~0.8 GB | community int8 encoder, faster on CPU, not gated |
| `fp32` | `ai4bharat/indic-conformer-600m-multilingual` | ~2.6 GB | official; **gated**: accept terms on the HF page + set a token |

Only the Bengali head is downloaded. Decoding: **CTC** (fast) and **RNNT** (often a bit more accurate, slower).

> **RTF** = processing time / audio duration. Below 1.0 is faster than real time.
> Use **Runtime > Change runtime type > CPU** to measure CPU-server speed.

## 1. Settings

In [ ]:
VARIANT = "int8"          # "int8" or "fp32"
DECODING = "both"         # "ctc", "rnnt" or "both"
THREADS = 0               # CPU threads, 0 = all cores
CHUNK_SEC = 30            # split long audio into N-second chunks (0 = no chunking)

AUDIO_SOURCE = "upload"   # "upload" = pick files from your PC, "drive" = Google Drive folder
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/sample-audio"

# fp32 only: add a secret named HF_TOKEN in Colab (key icon on the left), or paste it here
HF_TOKEN = ""

## 2. Install dependencies (~1 minute)

In [ ]:
!apt-get -qq install -y ffmpeg libsndfile1 > /dev/null
!pip install -q onnxruntime huggingface_hub librosa soundfile jiwer

## 3. Imports and environment check

In [ ]:
import os, sys, time, json, shutil, traceback
from pathlib import Path
import numpy as np, torch, librosa, onnxruntime as ort

if THREADS:
    torch.set_num_threads(THREADS)
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("onnxruntime :", ort.__version__)
print("cpu cores   :", os.cpu_count())
!lscpu | grep -E "Model name|^CPU\(s\)"

## 4. Download the model (Bengali head only)

In [ ]:
os.environ["HF_HUB_DISABLE_XET"] = "1"   # Xet transfer can stall; plain HTTP is reliable
from huggingface_hub import snapshot_download

REPOS = {"int8": "yashwork-byte/indic-conformer-600m-int8-onnx",
         "fp32": "ai4bharat/indic-conformer-600m-multilingual"}
OTHER_LANGS = ["as","brx","doi","gu","hi","kn","kok","ks","mai","ml","mni",
               "mr","ne","or","pa","sa","sat","sd","ta","te","ur"]

token = HF_TOKEN or None
if VARIANT == "fp32" and not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception as e:
        print("No HF_TOKEN secret found:", e)

MODEL_DIR = Path(f"/content/model/{VARIANT}")
t0 = time.time()
snapshot_download(REPOS[VARIANT], local_dir=MODEL_DIR, token=token,
                  allow_patterns=["assets/*", "config.json", "README.md"],
                  ignore_patterns=[f"assets/joint_post_net_{l}.onnx" for l in OTHER_LANGS])
print(f"Downloaded in {time.time()-t0:.0f}s")
!du -sh {MODEL_DIR}

## 5. Runner (same code as `indic_conformer_600m_bn/indic_conformer.py`)

In [ ]:
"""
Minimal offline runner for AI4Bharat IndicConformer-600M-multilingual (ONNX export).

Adapted from the model's own `model_onnx.py` (MIT, AI4Bharat) with these changes:
  * loads from a local folder only (no network, no trust_remote_code)
  * loads just the Bengali head instead of all 22 languages (faster start, less RAM)
  * CPU thread count is configurable
  * RNNT parts are only loaded when RNNT decoding is requested

Pipeline:  wav (16 kHz mono) -> preprocessor.ts (log-mel) -> encoder.onnx
           -> CTC:  ctc_decoder.onnx -> greedy collapse
           -> RNNT: rnnt_decoder + joint_* -> greedy transducer search
"""

import json
import warnings
from pathlib import Path

import numpy as np
import onnxruntime as ort
import torch

BLANK_ID = 256
SOS = 256
RNNT_MAX_SYMBOLS = 10
PRED_RNN_LAYERS = 2
PRED_RNN_HIDDEN_DIM = 640


class IndicConformerONNX:
    def __init__(self, model_dir, lang: str = "bn", threads: int = 0, load_rnnt: bool = True):
        self.assets = Path(model_dir) / "assets"
        if not (self.assets / "encoder.onnx").exists():
            raise FileNotFoundError(f"encoder.onnx not found in {self.assets}. Run download_model.py")
        self.lang = lang

        so = ort.SessionOptions()
        if threads:
            so.intra_op_num_threads = threads
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self._so = so

        with warnings.catch_warnings():  # torch.jit.load deprecation notice is harmless here
            warnings.simplefilter("ignore", FutureWarning)
            self.preprocessor = torch.jit.load(str(self.assets / "preprocessor.ts"), map_location="cpu")
        self.preprocessor.eval()
        self.encoder = self._session("encoder")
        self.ctc_decoder = self._session("ctc_decoder")

        self.vocab = json.loads((self.assets / "vocab.json").read_text(encoding="utf-8"))[lang]
        masks = json.loads((self.assets / "language_masks.json").read_text(encoding="utf-8"))
        self.mask = np.asarray(masks[lang], dtype=bool)

        self.rnnt_loaded = False
        if load_rnnt:
            self.load_rnnt()

    def _session(self, name: str) -> ort.InferenceSession:
        return ort.InferenceSession(str(self.assets / f"{name}.onnx"), sess_options=self._so,
                                    providers=["CPUExecutionProvider"])

    def load_rnnt(self):
        if self.rnnt_loaded:
            return
        self.rnnt_decoder = self._session("rnnt_decoder")
        self.joint_enc = self._session("joint_enc")
        self.joint_pred = self._session("joint_pred")
        self.joint_pre_net = self._session(f"joint_pre_net")
        self.joint_post_net = self._session(f"joint_post_net_{self.lang}")
        self.rnnt_loaded = True

    # ------------------------------------------------------------------ #
    @torch.inference_mode()
    def encode(self, wav: np.ndarray):
        """wav: float32 mono 16 kHz numpy array."""
        x = torch.from_numpy(np.ascontiguousarray(wav, dtype=np.float32)).unsqueeze(0)
        feats, length = self.preprocessor(input_signal=x, length=torch.tensor([x.shape[-1]]))
        out, enc_len = self.encoder.run(["outputs", "encoded_lengths"],
                                        {"audio_signal": feats.numpy(), "length": length.numpy()})
        return out, enc_len

    def transcribe(self, wav: np.ndarray, decoding: str = "ctc") -> str:
        enc, enc_len = self.encode(wav)
        if decoding == "ctc":
            return self._ctc(enc)
        if decoding == "rnnt":
            self.load_rnnt()
            return self._rnnt(enc)
        raise ValueError(f"decoding must be 'ctc' or 'rnnt', got {decoding!r}")

    # ------------------------------------------------------------------ #
    def _ctc(self, enc) -> str:
        logits = self.ctc_decoder.run(["logprobs"], {"encoder_output": enc})[0]
        logits = logits[:, :, self.mask]                       # keep Bengali tokens only
        ids = logits[0].argmax(axis=-1)
        # collapse repeats, drop blanks
        keep = np.ones_like(ids, dtype=bool)
        keep[1:] = ids[1:] != ids[:-1]
        ids = ids[keep]
        return "".join(self.vocab[i] for i in ids if i != BLANK_ID).replace("▁", " ").strip()

    def _rnnt(self, enc) -> str:
        f_all = self.joint_enc.run(["output"], {"input": enc.transpose(0, 2, 1)})[0]
        hyp = [SOS]
        state = (np.zeros((PRED_RNN_LAYERS, 1, PRED_RNN_HIDDEN_DIM), dtype=np.float32),
                 np.zeros((PRED_RNN_LAYERS, 1, PRED_RNN_HIDDEN_DIM), dtype=np.float32))
        g_cache = {}

        for t in range(f_all.shape[1]):
            f = f_all[:, t:t + 1, :]
            for _ in range(RNNT_MAX_SYMBOLS):
                key = (hyp[-1], len(hyp))
                if key not in g_cache:  # prediction net only changes when a token is emitted
                    g, _, s0, s1 = self.rnnt_decoder.run(
                        ["outputs", "prednet_lengths", "states", "162"],
                        {"targets": np.array([[hyp[-1]]], dtype=np.int32),
                         "target_length": np.array([1], dtype=np.int32),
                         "states.1": state[0], "onnx::Slice_3": state[1]})
                    g = self.joint_pred.run(["output"], {"input": g.transpose(0, 2, 1)})[0]
                    g_cache = {key: (g, s0, s1)}
                g, s0, s1 = g_cache[key]

                joint = self.joint_pre_net.run(["output"], {"input": f + g})[0]
                logits = self.joint_post_net.run(["output"], {"input": joint})[0]
                tok = int(logits.reshape(-1).argmax())
                if tok == BLANK_ID:
                    break
                hyp.append(tok)
                state = (s0, s1)

        return "".join(self.vocab[i] for i in hyp[1:]).replace("▁", " ").strip()

## 6. Load the model

In [ ]:
t0 = time.time()
model = IndicConformerONNX(MODEL_DIR, lang="bn", threads=THREADS,
                           load_rnnt=DECODING in ("rnnt", "both"))
LOAD_SEC = time.time() - t0
print(f"Model loaded in {LOAD_SEC:.1f}s")

## 7. Get your audio
* **upload**: select your audio files, plus optional reference transcripts named like the
  audio (`call.mp3` + `call.txt`) to get WER/CER.
* **drive**: mounts Google Drive and reads `DRIVE_AUDIO_DIR`.

In [ ]:
AUDIO_DIR = Path("/content/sample-audio")
AUDIO_DIR.mkdir(exist_ok=True)
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus", ".webm"}

if AUDIO_SOURCE == "upload":
    from google.colab import files
    for name, data in files.upload().items():
        (AUDIO_DIR / name).write_bytes(data)
elif AUDIO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    for p in Path(DRIVE_AUDIO_DIR).iterdir():
        if p.suffix.lower() in AUDIO_EXTS | {".txt"}:
            shutil.copy(p, AUDIO_DIR / p.name)

AUDIO_FILES = sorted(p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in AUDIO_EXTS)
print(f"{len(AUDIO_FILES)} audio file(s):")
for p in AUDIO_FILES:
    print(" -", p.name + (" (+ reference .txt)" if p.with_suffix(".txt").exists() else ""))

## 8. Run the test

In [ ]:
import jiwer
SR = 16000
decodings = ["ctc", "rnnt"] if DECODING == "both" else [DECODING]

def split_chunks(audio, chunk_sec):
    size = int(chunk_sec * SR)
    if size <= 0 or len(audio) <= size:
        return [audio]
    chunks = [audio[i:i + size] for i in range(0, len(audio), size)]
    if len(chunks) > 1 and len(chunks[-1]) < SR:
        chunks[-2] = np.concatenate([chunks[-2], chunks.pop()])
    return chunks

results, errors = [], []
for f in AUDIO_FILES:
    print("=" * 80)
    print("FILE:", f.name)
    rec = {"file": f.name, "status": "ok"}
    try:
        audio, _ = librosa.load(str(f), sr=SR, mono=True)
        if audio.size == 0:
            raise ValueError("audio decoded to 0 samples")
        dur = len(audio) / SR
        chunks = split_chunks(audio, CHUNK_SEC)
        rec.update(duration_sec=round(dur, 2), chunks=len(chunks))
        print(f"duration {dur:.2f}s | chunks {len(chunks)}")
        ref_file = f.with_suffix(".txt")
        ref = ref_file.read_text(encoding="utf-8").strip() if ref_file.exists() else None

        for dec in decodings:
            t0 = time.time()
            text = " ".join(t for t in (model.transcribe(c, dec) for c in chunks) if t)
            infer = time.time() - t0
            rec[f"{dec}_sec"] = round(infer, 2)
            rec[f"{dec}_rtf"] = round(infer / dur, 3)
            rec[f"{dec}_text"] = text
            print(f"\n[{dec.upper()}] inference {infer:.2f}s | RTF {infer/dur:.3f}")
            print(text or "<empty>")
            if ref:
                rec[f"{dec}_wer"] = round(jiwer.wer(ref, text), 4)
                rec[f"{dec}_cer"] = round(jiwer.cer(ref, text), 4)
                print(f"WER {rec[f'{dec}_wer']:.2%} | CER {rec[f'{dec}_cer']:.2%}")
    except Exception as e:
        rec.update(status="error", error=f"{type(e).__name__}: {e}")
        errors.append((f.name, traceback.format_exc()))
        print("ERROR:", rec["error"])
    results.append(rec)

## 9. Summary

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
df = pd.DataFrame(results)
display(df[[c for c in df.columns if not c.endswith("_text")]])

ok = df[df.status == "ok"]
print(f"Model load {LOAD_SEC:.1f}s | {len(ok)} ok, {len(errors)} failed")
for dec in decodings:
    if len(ok):
        print(f"[{dec.upper()}] overall RTF {ok[f'{dec}_sec'].sum()/ok.duration_sec.sum():.3f}")

## 10. Errors (full tracebacks)

In [ ]:
if not errors:
    print("No errors")
for name, tb in errors:
    print("=" * 80)
    print("FILE:", name)
    print(tb)

## 11. Save and download results

In [ ]:
from datetime import datetime
out_json = f"/content/indic_conformer_600m_bn_{VARIANT}_results_{datetime.now():%Y%m%d_%H%M%S}.json"
with open(out_json, "w", encoding="utf-8") as fh:
    json.dump({"model": REPOS[VARIANT], "variant": VARIANT, "decoding": decodings,
               "threads": torch.get_num_threads(), "chunk_sec": CHUNK_SEC,
               "load_sec": round(LOAD_SEC, 2), "results": results,
               "errors": [{"file": n, "traceback": t} for n, t in errors]},
              fh, ensure_ascii=False, indent=2)
print("Saved", out_json)
from google.colab import files
files.download(out_json)